# 🏋️ Module 04: Training, Validation & Comprehensive Evaluation
### Multi-Model Training Loop, Early Stopping & Unseen Test Evaluation

This notebook implements:
1. Complete training routine with AdamW optimizer, L2 weight decay, and ReduceLROnPlateau.
2. Early Stopping with best-weight restoration.
3. Loading pre-trained model weights from cache or training on demand.
4. Comprehensive multi-metric evaluation: **Accuracy, Macro Precision, Macro Recall, Macro F1, Weighted F1**.
5. Confusion matrices and grouped benchmark visualization charts.

In [ ]:
import os
import copy
import time
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

from src.dataset import load_kermany_oct_data, get_oct_dataloaders, CLASS_NAMES
from src.models import build_model, SUPPORTED_MODELS

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Compute Device: {device}")

### 4.1 Training Functions with Early Stopping

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels.data).item()
        total += labels.size(0)

    return running_loss / total, (correct / total) * 100.0


def validate_one_epoch(model, dataloader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct += torch.sum(preds == labels.data).item()
            total += labels.size(0)

    return running_loss / total, (correct / total) * 100.0


def train_model(
    model, train_loader, val_loader, num_epochs=10, lr=1e-3, weight_decay=1e-4, patience=3, device=None, model_name="Model"
):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "lr": []}
    best_val_loss = float("inf")
    best_weights = copy.deepcopy(model.state_dict())
    epochs_no_improve = 0

    print(f"\n--- Training {model_name} on {device} (Epochs: {num_epochs}, LR: {lr}) ---")
    start_time = time.time()

    for epoch in range(num_epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = validate_one_epoch(model, val_loader, criterion, device)
        current_lr = optimizer.param_groups[0]["lr"]
        scheduler.step(val_loss)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["lr"].append(current_lr)

        print(f"Epoch {epoch+1:02d}/{num_epochs:02d} | Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}% | LR: {current_lr:.6f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping triggered at epoch {epoch+1} (No improvement for {patience} epochs).")
                break

    print(f"Training completed in {time.time() - start_time:.2f}s. Restoring best weights (Val Loss: {best_val_loss:.4f}).")
    model.load_state_dict(best_weights)
    return model, history

### 4.2 Evaluation Function

In [ ]:
def evaluate_model(model, test_loader, device=None, class_names=None):
    if class_names is None: class_names = CLASS_NAMES
    if device is None: device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model.eval()
    model.to(device)

    all_preds, all_targets, all_probs = [], [], []
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            probs = F.softmax(outputs, dim=1)
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
            all_targets.extend(targets.numpy())

    all_preds, all_targets, all_probs = np.array(all_preds), np.array(all_targets), np.array(all_probs)

    acc = float(accuracy_score(all_targets, all_preds))
    prec_macro = float(precision_score(all_targets, all_preds, average="macro", zero_division=0))
    rec_macro = float(recall_score(all_targets, all_preds, average="macro", zero_division=0))
    f1_macro = float(f1_score(all_targets, all_preds, average="macro", zero_division=0))

    cm = confusion_matrix(all_targets, all_preds)
    report_str = classification_report(all_targets, all_preds, target_names=class_names, zero_division=0)

    misclassified_indices = np.where(all_preds != all_targets)[0]

    return {
        "accuracy": acc,
        "precision_macro": prec_macro,
        "recall_macro": rec_macro,
        "f1_macro": f1_macro,
        "confusion_matrix": cm,
        "classification_report": report_str,
        "num_test_samples": len(all_targets),
        "num_misclassified": len(misclassified_indices)
    }

### 4.3 Training & Evaluating All 5 Models

In [ ]:
splits = load_kermany_oct_data()
train_loader, val_loader, test_loader = get_oct_dataloaders(splits, batch_size=32)

models_to_run = [
    ("vgg16", {"epochs": 3, "lr": 1e-3, "l2": 1e-4, "freeze": True}),
    ("resnet18", {"epochs": 4, "lr": 1e-3, "l2": 1e-4, "freeze": True}),
    ("densenet121", {"epochs": 4, "lr": 1e-3, "l2": 1e-4, "freeze": True}),
    ("mobilenet_v2", {"epochs": 4, "lr": 1e-3, "l2": 1e-4, "freeze": True}),
    ("custom_cnn", {"epochs": 6, "lr": 1e-3, "l2": 1e-4, "freeze": False}),
]

trained_models = {}
eval_metrics = {}
confusion_matrices = {}
benchmark_records = []

for model_name, cfg in models_to_run:
    pkl_path = f"models/{model_name}.pkl"
    loaded = False
    if os.path.exists(pkl_path):
        try:
            with open(pkl_path, "rb") as f:
                pkg = pickle.load(f)
            model = pkg["model"]
            print(f"Loaded cached {model_name.upper()} from {pkl_path}")
            loaded = True
        except Exception as e:
            print(f"Could not load {pkl_path}: {e}")

    if not loaded:
        print(f"Training {model_name.upper()}...")
        model = build_model(model_name, num_classes=len(CLASS_NAMES), pretrained=True, freeze_backbone=cfg["freeze"])
        model, hist = train_model(model, train_loader, val_loader, num_epochs=cfg["epochs"], lr=cfg["lr"], weight_decay=cfg["l2"], device=device, model_name=model_name.upper())

    metrics = evaluate_model(model, test_loader, device=device, class_names=CLASS_NAMES)
    trained_models[model_name] = model
    eval_metrics[model_name] = metrics
    confusion_matrices[model_name] = metrics["confusion_matrix"]

    benchmark_records.append({
        "Model": model_name.upper(),
        "Accuracy": metrics["accuracy"],
        "Precision": metrics["precision_macro"],
        "Recall": metrics["recall_macro"],
        "F1-Score": metrics["f1_macro"],
        "Misclassified": metrics["num_misclassified"]
    })

benchmark_df = pd.DataFrame(benchmark_records).sort_values(by="F1-Score", ascending=False)
print("\n" + "=" * 65)
print("BENCHMARK COMPARISON TABLE (TEST SPLIT):")
print("=" * 65)
print(benchmark_df.to_string(index=False))
print("=" * 65)

### 4.4 Visualizing Confusion Matrices & Comparison Bar Chart

In [ ]:
# Confusion Matrices
cols = 3
rows = (len(confusion_matrices) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(5.5 * cols, 5 * rows))
axes = axes.flatten()

for idx, (m_name, cm) in enumerate(confusion_matrices.items()):
    ax = axes[idx]
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title(f"{m_name.upper()} (Test Set)", fontsize=12, fontweight="bold")
    ax.set_xlabel("Predicted Condition", fontsize=10)
    ax.set_ylabel("True Condition", fontsize=10)

for j in range(len(confusion_matrices), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
# Benchmark Comparison Bar Chart
fig, ax = plt.subplots(figsize=(11, 6))
model_list = benchmark_df["Model"].tolist()
metrics_keys = ["Accuracy", "Precision", "Recall", "F1-Score"]
x = np.arange(len(model_list))
width = 0.2
colors = ["#1f77b4", "#2ca02c", "#ff7f0e", "#9467bd"]

for i, metric in enumerate(metrics_keys):
    values = [benchmark_df.loc[benchmark_df["Model"] == m, metric].values[0] for m in model_list]
    rects = ax.bar(x + (i - 1.5) * width, values, width, label=metric, color=colors[i], alpha=0.9)
    for rect in rects:
        h = rect.get_height()
        ax.annotate(f"{h:.2f}", xy=(rect.get_x() + rect.get_width() / 2, h),
                    xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontsize=8)

ax.set_ylabel("Score (0.0 - 1.0)", fontsize=11, fontweight="bold")
ax.set_title("Comprehensive Model Benchmark Comparison (Test Set)", fontsize=13, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(model_list, fontsize=10, fontweight="bold")
ax.set_ylim(0, 1.15)
ax.legend(loc="upper left")
ax.grid(axis="y", linestyle=":", alpha=0.7)
plt.tight_layout()
plt.show()